# A2 — Optimization with Genetic Algorithms (Graph Coloring)

This notebook implements a **Genetic Algorithm (GA)** to solve the **Graph Coloring Problem (GCP)** with:
- **2 selection methods** (tournament, roulette)
- **2 crossover methods** (one-point, uniform)
- **2 mutation methods** (random reset, greedy)

It also includes an **experiment runner** to execute **6 parameter combinations** per dataset and produce:
- JSON summaries
- A fitness evolution plot for the best run

Dataset format: DIMACS-like `.col` files.

> Assignment reference: see uploaded PDF. fileciteturn0file0


In [ ]:
pip install numpy 


In [ ]:
pip install matplotlib


In [ ]:
# If running in a fresh environment:
# !pip install numpy matplotlib

import os
import json
import random
import time
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt


: 

## 1) Load Graph (.col) + adjacency

In [ ]:
def load_col_graph(path: str) -> Tuple[int, List[Tuple[int, int]]]:
    """Load a graph from DIMACS-like .col files.
    
    Expected lines:
      - p edge V E
      - e u v  (1-indexed vertices)
    
    Returns:
      V (int), edges (list of (u, v) 0-indexed)
    """
    V = None
    edges: List[Tuple[int, int]] = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("c"):
                continue
            parts = line.split()
            if parts[0] == "p":
                V = int(parts[2])
            elif parts[0] == "e":
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                if u != v:
                    edges.append((u, v))
    if V is None:
        raise ValueError(f"Could not find 'p edge V E' line in {path}")
    return V, edges


def build_adjacency(V: int, edges: List[Tuple[int, int]]) -> List[List[int]]:
    adj = [[] for _ in range(V)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj


## 2) GA configuration + result types

In [ ]:
@dataclass
class GAConfig:
    population_size: int
    generations: int
    elite_count: int = 1

    selection: str = "tournament"   # tournament | roulette
    crossover: str = "one_point"    # one_point | uniform
    mutation: str = "random_reset"  # random_reset | greedy

    crossover_rate: float = 0.9
    mutation_rate: float = 0.05
    tournament_k: int = 3

    conflict_weight: float = 50.0   # alpha in alpha*conflicts + colors_used

    patience: int = 100            # stop if no improvement for this many generations
    max_colors: Optional[int] = None  # if None -> V
    seed: Optional[int] = 42


@dataclass
class GAResult:
    best_chromosome: np.ndarray
    best_fitness: float
    best_conflicts: int
    best_colors_used: int
    fitness_history: List[float]
    time_seconds: float
    generations_ran: int
    config: GAConfig


## 3) Fitness: minimize conflicts, then colors

Fitness to **minimize**:
\[ f = \alpha \cdot \text{conflicts} + \text{colors\_used} \]


In [ ]:
def count_conflicts(chrom: np.ndarray, edges: List[Tuple[int, int]]) -> int:
    conflicts = 0
    for u, v in edges:
        if chrom[u] == chrom[v]:
            conflicts += 1
    return conflicts


def colors_used(chrom: np.ndarray) -> int:
    return int(len(np.unique(chrom)))


def fitness(chrom: np.ndarray, edges: List[Tuple[int, int]], conflict_weight: float) -> float:
    c = count_conflicts(chrom, edges)
    k = colors_used(chrom)
    return conflict_weight * c + k


## 4) GA operators (2× selection, 2× crossover, 2× mutation)

In [ ]:
def init_population(pop_size: int, V: int, K: int, rng: random.Random) -> np.ndarray:
    return np.array([[rng.randrange(K) for _ in range(V)] for _ in range(pop_size)], dtype=np.int32)


# --- Selection (2 required) ---
def select_parent_tournament(pop: np.ndarray, fit: np.ndarray, k: int, rng: random.Random) -> np.ndarray:
    idxs = [rng.randrange(len(pop)) for _ in range(k)]
    best_i = min(idxs, key=lambda i: fit[i])
    return pop[best_i].copy()


def select_parent_roulette(pop: np.ndarray, fit: np.ndarray, rng: random.Random) -> np.ndarray:
    # lower fitness is better -> use inverse fitness as weight
    inv = 1.0 / (fit + 1e-9)
    probs = inv / inv.sum()
    r = rng.random()
    cum = 0.0
    for i, p in enumerate(probs):
        cum += p
        if r <= cum:
            return pop[i].copy()
    return pop[-1].copy()


# --- Crossover (2 required) ---
def crossover_one_point(p1: np.ndarray, p2: np.ndarray, rng: random.Random):
    V = len(p1)
    if V < 2:
        return p1.copy(), p2.copy()
    cut = rng.randrange(1, V)
    c1 = np.concatenate([p1[:cut], p2[cut:]]).astype(np.int32)
    c2 = np.concatenate([p2[:cut], p1[cut:]]).astype(np.int32)
    return c1, c2


def crossover_uniform(p1: np.ndarray, p2: np.ndarray, rng: random.Random):
    V = len(p1)
    mask = np.array([rng.random() < 0.5 for _ in range(V)], dtype=bool)
    c1 = p1.copy()
    c2 = p2.copy()
    c1[mask] = p2[mask]
    c2[mask] = p1[mask]
    return c1.astype(np.int32), c2.astype(np.int32)


# --- Mutation (2 required) ---
def mutate_random_reset(chrom: np.ndarray, K: int, mutation_rate: float, rng: random.Random) -> np.ndarray:
    out = chrom.copy()
    for i in range(len(out)):
        if rng.random() < mutation_rate:
            out[i] = rng.randrange(K)
    return out


def mutate_greedy(chrom: np.ndarray, adj: List[List[int]], K: int, mutation_rate: float, rng: random.Random) -> np.ndarray:
    out = chrom.copy()
    V = len(out)
    for v in range(V):
        if rng.random() >= mutation_rate:
            continue
        cv = out[v]
        conflicted = any(out[n] == cv for n in adj[v])
        if not conflicted:
            continue

        best_color = cv
        best_local = 10**9
        for c in range(K):
            local = 0
            for n in adj[v]:
                if out[n] == c:
                    local += 1
            if local < best_local:
                best_local = local
                best_color = c
        out[v] = best_color
    return out


## 5) GA loop with elitism + stationary detection

In [ ]:
def run_ga(V: int, edges: List[Tuple[int, int]], cfg: GAConfig) -> GAResult:
    if cfg.seed is not None:
        rng = random.Random(cfg.seed)
        np.random.seed(cfg.seed)
    else:
        rng = random.Random()

    adj = build_adjacency(V, edges)
    K = cfg.max_colors if cfg.max_colors is not None else V

    pop = init_population(cfg.population_size, V, K, rng)
    fit = np.array([fitness(ind, edges, cfg.conflict_weight) for ind in pop], dtype=np.float64)

    best_idx = int(np.argmin(fit))
    best = pop[best_idx].copy()
    best_fit = float(fit[best_idx])
    best_conf = count_conflicts(best, edges)
    best_k = colors_used(best)

    history = [best_fit]
    no_improve = 0
    start = time.time()

    for gen in range(cfg.generations):
        elite_idxs = np.argsort(fit)[: cfg.elite_count]
        new_pop = [pop[i].copy() for i in elite_idxs]

        while len(new_pop) < cfg.population_size:
            # selection
            if cfg.selection == "tournament":
                p1 = select_parent_tournament(pop, fit, cfg.tournament_k, rng)
                p2 = select_parent_tournament(pop, fit, cfg.tournament_k, rng)
            elif cfg.selection == "roulette":
                p1 = select_parent_roulette(pop, fit, rng)
                p2 = select_parent_roulette(pop, fit, rng)
            else:
                raise ValueError(f"Unknown selection: {cfg.selection}")

            # crossover
            if rng.random() < cfg.crossover_rate:
                if cfg.crossover == "one_point":
                    c1, c2 = crossover_one_point(p1, p2, rng)
                elif cfg.crossover == "uniform":
                    c1, c2 = crossover_uniform(p1, p2, rng)
                else:
                    raise ValueError(f"Unknown crossover: {cfg.crossover}")
            else:
                c1, c2 = p1.copy(), p2.copy()

            # mutation
            if cfg.mutation == "random_reset":
                c1 = mutate_random_reset(c1, K, cfg.mutation_rate, rng)
                c2 = mutate_random_reset(c2, K, cfg.mutation_rate, rng)
            elif cfg.mutation == "greedy":
                c1 = mutate_greedy(c1, adj, K, cfg.mutation_rate, rng)
                c2 = mutate_greedy(c2, adj, K, cfg.mutation_rate, rng)
            else:
                raise ValueError(f"Unknown mutation: {cfg.mutation}")

            new_pop.append(c1)
            if len(new_pop) < cfg.population_size:
                new_pop.append(c2)

        pop = np.array(new_pop, dtype=np.int32)
        fit = np.array([fitness(ind, edges, cfg.conflict_weight) for ind in pop], dtype=np.float64)

        gen_best_idx = int(np.argmin(fit))
        gen_best = pop[gen_best_idx].copy()
        gen_best_fit = float(fit[gen_best_idx])

        if gen_best_fit + 1e-12 < best_fit:
            best_fit = gen_best_fit
            best = gen_best.copy()
            best_conf = count_conflicts(best, edges)
            best_k = colors_used(best)
            no_improve = 0
        else:
            no_improve += 1

        history.append(best_fit)

        if no_improve >= cfg.patience:
            end = time.time()
            return GAResult(
                best_chromosome=best,
                best_fitness=best_fit,
                best_conflicts=best_conf,
                best_colors_used=best_k,
                fitness_history=history,
                time_seconds=end - start,
                generations_ran=gen + 1,
                config=cfg,
            )

    end = time.time()
    return GAResult(
        best_chromosome=best,
        best_fitness=best_fit,
        best_conflicts=best_conf,
        best_colors_used=best_k,
        fitness_history=history,
        time_seconds=end - start,
        generations_ran=cfg.generations,
        config=cfg,
    )


def pretty_solution_summary(res: GAResult) -> str:
    cfg = res.config
    return (
        f"Best fitness={res.best_fitness:.3f} | conflicts={res.best_conflicts} | colors_used={res.best_colors_used} | " 
        f"gens={res.generations_ran} | time={res.time_seconds:.2f}s | sel={cfg.selection} | cross={cfg.crossover} | "
        f"mut={cfg.mutation} | pop={cfg.population_size} | cr={cfg.crossover_rate} | mr={cfg.mutation_rate} | alpha={cfg.conflict_weight}"
    )


## 6) Single-run demo (edit `DATASET_PATH`)

Download a `.col` file and set `DATASET_PATH` accordingly.
Recommended datasets are on the CMU COLOR instances page (linked in the assignment PDF). fileciteturn0file0


In [ ]:
# --- Set this to your downloaded dataset path ---
DATASET_PATH = "data/myciel3.col"  # <- change me

# Load
# V, edges = load_col_graph(DATASET_PATH)
# print("V:", V, "E:", len(edges))

# Example config
# cfg = GAConfig(
#     population_size=min(500, max(80, 10 * V)),
#     generations=2000,
#     elite_count=2,
#     selection="tournament",
#     crossover="uniform",
#     mutation="greedy",
#     crossover_rate=0.9,
#     mutation_rate=0.05,
#     tournament_k=3,
#     conflict_weight=50.0,
#     patience=150,
#     max_colors=V,
#     seed=42,
# )

# res = run_ga(V, edges, cfg)
# print(pretty_solution_summary(res))

# Plot fitness evolution
# plt.figure()
# plt.plot(res.fitness_history)
# plt.xlabel("Generation")
# plt.ylabel("Best fitness so far")
# plt.title("Fitness evolution (best so far)")
# plt.show()


## 7) Experiments: 6 operator combinations + outputs

This matches the assignment requirement to run multiple parameter combinations and show fitness evolution for the best run. fileciteturn0file0


In [ ]:
def ensure_dir(d: str) -> None:
    os.makedirs(d, exist_ok=True)


def save_fitness_plot(history, out_path: str, title: str) -> None:
    plt.figure()
    plt.plot(history)
    plt.xlabel("Generation")
    plt.ylabel("Best fitness so far")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def run_6_combo_experiments(tag: str, dataset_path: str, out_dir: str = "results"):
    V, edges = load_col_graph(dataset_path)
    pop_size = min(500, max(80, 10 * V))

    combos = [
        ("tournament", "one_point", "random_reset"),
        ("tournament", "uniform", "random_reset"),
        ("tournament", "one_point", "greedy"),
        ("roulette", "one_point", "random_reset"),
        ("roulette", "uniform", "random_reset"),
        ("roulette", "one_point", "greedy"),
    ]

    base = dict(
        generations=2000,
        elite_count=2,
        crossover_rate=0.9,
        mutation_rate=0.05,
        tournament_k=3,
        conflict_weight=50.0,
        patience=150,
        seed=42,
    )

    dataset_dir = os.path.join(out_dir, tag)
    ensure_dir(dataset_dir)

    summary_rows = []
    best_run = None

    for i, (sel, cross, mut) in enumerate(combos, start=1):
        cfg = GAConfig(
            population_size=pop_size,
            selection=sel,
            crossover=cross,
            mutation=mut,
            max_colors=V,
            **base,
        )
        res = run_ga(V, edges, cfg)

        print(f"[{tag}] run {i}/6:", pretty_solution_summary(res))

        row = {
            "dataset": tag,
            "path": dataset_path,
            "V": V,
            "E": len(edges),
            "run": i,
            "selection": sel,
            "crossover": cross,
            "mutation": mut,
            "pop_size": pop_size,
            "generations_ran": res.generations_ran,
            "time_seconds": res.time_seconds,
            "best_fitness": res.best_fitness,
            "best_conflicts": res.best_conflicts,
            "best_colors_used": res.best_colors_used,
        }
        summary_rows.append(row)

        with open(os.path.join(dataset_dir, f"run_{i}.json"), "w", encoding="utf-8") as f:
            json.dump(row, f, indent=2)

        if best_run is None or res.best_fitness < best_run.best_fitness:
            best_run = res

    with open(os.path.join(dataset_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary_rows, f, indent=2)

    if best_run is not None:
        plot_path = os.path.join(dataset_dir, "best_fitness_evolution.png")
        save_fitness_plot(
            best_run.fitness_history,
            plot_path,
            title=f"{tag}: best fitness evolution (V={V}, E={len(edges)})",
        )
        print(f"[{tag}] best plot saved -> {plot_path}")

    return summary_rows


# Example (uncomment after downloading datasets):
# run_6_combo_experiments("small", "data/myciel3.col")
# run_6_combo_experiments("medium", "data/queen7_7.col")
# run_6_combo_experiments("large", "data/dsjc500.1.col")
